# AI-Powered Finance Policy Assistant
## Retrieval-Augmented Generation (RAG) System

---

### Overview
This notebook implements a production-grade **Finance Policy Assistant** using RAG (Retrieval-Augmented Generation). Employees can ask natural-language questions about company finance policies and receive accurate, grounded answers with source references.

---

### RAG Pipeline Architecture

```
╔══════════════════════════════════════════════════════════════════════════╗
║              INGESTION PIPELINE (One-time setup)                        ║
╠══════════════════════════════════════════════════════════════════════════╣
║                                                                          ║
║  Finance Policy     RecursiveCharacter    text-embedding-      FAISS     ║
║  Documents (6)  ──► TextSplitter      ──► 3-small (OpenAI) ──► Vector   ║
║  (Raw Text)         chunk_size=500        Embeddings           Store     ║
║                     chunk_overlap=50                                      ║
║                                                                          ║
╠══════════════════════════════════════════════════════════════════════════╣
║              QUERY PIPELINE (Real-time inference)                        ║
╠══════════════════════════════════════════════════════════════════════════╣
║                                                                          ║
║  User Query    text-embedding-    Similarity      Retrieved              ║
║  (Natural  ──► 3-small        ──► Search      ──► Context   ──►          ║
║  Language)     Embeddings         (Top-4 chunks)  (Policy                ║
║                                                   Snippets)              ║
║                                                       │                  ║
║                                                       ▼                  ║
║                                                  GPT-4o-mini             ║
║                                                  (LLM Reasoning)         ║
║                                                       │                  ║
║                                                       ▼                  ║
║                                            Grounded Answer +             ║
║                                            Source References             ║
║                                                                          ║
╚══════════════════════════════════════════════════════════════════════════╝
```

### Tech Stack
| Component | Technology |
|-----------|------------|
| LLM | GPT-4o-mini (OpenAI) |
| Embeddings | text-embedding-3-small (OpenAI) |
| Orchestration | LangChain |
| Vector Database | FAISS (faiss-cpu) |
| Chunking | RecursiveCharacterTextSplitter |
| Visualization | Matplotlib + NetworkX |

In [ ]:
# Cell 2: Install all required dependencies
# This installs the latest compatible versions of all required packages
!pip install -qU langchain langchain-openai langchain-community faiss-cpu tiktoken matplotlib networkx

In [ ]:
# Cell 3: Import all required libraries

# ── LangChain core components ──────────────────────────────────────────────
from langchain.text_splitter import RecursiveCharacterTextSplitter  # type: ignore
from langchain.schema import Document  # type: ignore
from langchain.chains import RetrievalQA  # type: ignore
from langchain.prompts import PromptTemplate  # type: ignore

# ── LangChain OpenAI integration ───────────────────────────────────────────
from langchain_openai import ChatOpenAI, OpenAIEmbeddings  # type: ignore

# ── LangChain Community: FAISS vector store ────────────────────────────────
from langchain_community.vectorstores import FAISS  # type: ignore

# ── Visualization ──────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
import networkx as nx

# ── Standard library ───────────────────────────────────────────────────────
import textwrap
import datetime
import warnings
import os

# Suppress deprecation warnings for cleaner output
warnings.filterwarnings("ignore")

print("All libraries imported successfully!")
print(f"Session started at: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

In [ ]:
# Cell 4: API Key Setup + LLM + Embeddings Initialization
#
# PREREQUISITE: Before running this cell, add your OpenAI API key to
# Google Colab secrets (left sidebar → key icon) with name: OPENAI_API_KEY

from google.colab import userdata  # type: ignore

# Load API key from Colab secrets (secure — never hardcoded)
OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")

if not OPENAI_API_KEY:
    raise ValueError(
        "OPENAI_API_KEY not found in Colab secrets. "
        "Please add it via: left sidebar → Secrets (key icon) → Add new secret"
    )

print(f"API key loaded: sk-...{OPENAI_API_KEY[-4:]} (last 4 chars shown for verification)")

# ── Initialize LLM: GPT-4o-mini ───────────────────────────────────────────
# temperature=0 ensures deterministic, factual responses (no creativity/hallucination)
llm = ChatOpenAI(
    model="gpt-4o-mini",
    api_key=OPENAI_API_KEY,
    temperature=0
)

# ── Initialize Embeddings: text-embedding-3-small ─────────────────────────
# text-embedding-3-small: 1536-dim, cost-efficient, high quality for retrieval
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    api_key=OPENAI_API_KEY
)

print("LLM initialized: GPT-4o-mini (temperature=0)")
print("Embeddings initialized: text-embedding-3-small (1536 dimensions)")
print("Setup complete — ready to build the RAG pipeline!")

In [ ]:
# Cell 5: Finance Policy Synthetic Dataset
# Six detailed, realistic corporate finance policy documents.
# Each document is stored as a LangChain Document object with metadata.

# ─────────────────────────────────────────────────────────────────────────
# DOCUMENT 1: Employee Reimbursement Policy
# ─────────────────────────────────────────────────────────────────────────
doc1_text = """
EMPLOYEE REIMBURSEMENT POLICY
Policy Code: FIN-ERP-001 | Version: 3.2 | Effective Date: April 1, 2024
Issued by: Finance Controller | Approved by: CFO

1. PURPOSE AND SCOPE
This policy governs the reimbursement of business expenses incurred by employees
in the course of their official duties. It applies to all full-time employees,
part-time employees, and contractors working on company projects.

2. ELIGIBLE EXPENSES AND LIMITS

2.1 Client Entertainment
- Maximum limit: Rs. 5,000 per person per occasion
- Applicable for: Business meals, client hospitality, team events with clients
- Requires: Names of all attendees, business purpose, receipt
- Pre-approval required from Department Head for amounts above Rs. 3,000 per person
- Venue must be a recognized restaurant/hotel (no private clubs without prior approval)

2.2 Domestic Daily Allowance (Per Diem)
- Standard daily allowance: Rs. 3,000 per day
- Covers: Meals, local transportation, miscellaneous expenses
- Half-day rate applies when travel duration is less than 8 hours
- Per diem is not applicable if company provides accommodation and meals directly

2.3 International Daily Allowance
- Standard daily allowance: USD 100 per day
- Covers: Meals, local transportation, incidental expenses
- Additional allowances for high-cost cities (e.g., London, New York, Tokyo): USD 150/day
- SIM card / data roaming: Rs. 500 per day (maximum Rs. 5,000 per trip)

2.4 Communication Expenses
- Mobile phone bills: Up to Rs. 2,000/month for eligible employees (Grade 5 and above)
- Internet dongle/home broadband for remote work: Up to Rs. 1,500/month
- Requires itemized bill submission

2.5 Office Supplies
- Work-from-home office supplies: Up to Rs. 5,000 per year
- Requires original receipts and pre-approval from manager

3. RECEIPT AND DOCUMENTATION REQUIREMENTS
- Original receipts mandatory for all expenses above Rs. 500
- Digital/scanned copies accepted if original is lost (accompanied by self-declaration)
- GST invoice required for amounts above Rs. 5,000 to avail input tax credit
- All receipts must show: vendor name, date, amount, GST number (if applicable)
- Credit card statements alone are NOT accepted as receipts

4. SUBMISSION PROCESS AND DEADLINES
- Expense reports must be submitted within 30 days of expense incurrence
- Late submissions (31-60 days): Requires Finance Controller approval
- Submissions beyond 60 days will NOT be reimbursed without CFO approval
- Submit via: HRMS portal → Finance → Expense Reimbursement
- Attach all receipts as PDF/JPG files (max 5MB per file)

5. APPROVAL HIERARCHY
- Up to Rs. 5,000: Direct Manager approval
- Rs. 5,001 to Rs. 25,000: Department Head approval
- Rs. 25,001 and above: Finance Controller approval
- International expenses above USD 500: Both Department Head and Finance Controller

6. PAYMENT PROCESSING SLA
- Once fully approved: Payment processed within 7 working days
- Payment mode: Direct bank transfer to employee's registered salary account
- Payroll cycle: Claims approved before 20th of month are paid by month-end
- Claims approved after 20th: Paid in the following month's cycle

7. NON-REIMBURSABLE EXPENSES
The following expenses will NOT be reimbursed under any circumstances:
- Personal entertainment (movies, sporting events, personal dining)
- Alcohol and tobacco (unless specifically approved for client entertainment)
- Personal grooming, spa, gym memberships
- Traffic fines, parking violations, or penalties of any kind
- First-class or premium economy upgrades (unless medically certified)
- Expenses incurred during personal leave/vacation
- Gifts to employees or personal gifts (covered under separate gifting policy)
- Political donations or contributions
- Losses due to theft or negligence (must be reported separately)

8. FRAUD PREVENTION
- All claims are subject to audit by the Internal Audit team
- Misrepresentation of expenses will result in disciplinary action up to termination
- Duplicate submissions are detected through automated system checks
- Random 10% spot-check on all approved claims every quarter

9. CONTACT INFORMATION
- Email: finance-reimbursements@company.com
- Helpdesk extension: 4455
- Policy queries: finance-policy@company.com
"""

# ─────────────────────────────────────────────────────────────────────────
# DOCUMENT 2: Corporate Travel Expense Policy
# ─────────────────────────────────────────────────────────────────────────
doc2_text = """
CORPORATE TRAVEL EXPENSE POLICY
Policy Code: FIN-CTR-002 | Version: 2.8 | Effective Date: April 1, 2024
Issued by: Finance Department | Approved by: CFO & CHRO

1. PURPOSE
This policy establishes guidelines for business travel to ensure employee safety,
cost efficiency, and compliance with corporate financial controls.

2. TRAVEL AUTHORIZATION
- All domestic travel requires manager approval at least 48 hours in advance
- International travel requires Department Head + HR approval at least 7 days in advance
- Emergency travel (within 48 hours): Finance Controller verbal approval + written within 24 hrs
- Travel bookings must be done through company's approved travel management portal

3. AIR TRAVEL POLICY

3.1 Domestic Flights
- Class of travel: Economy class mandatory for all employees
- No exceptions to economy class for domestic travel regardless of grade/designation
- Baggage allowance: Covered up to airline's standard economy allowance only
- Seat upgrades: Not reimbursable; employee's personal expense
- Book at least 7 days in advance to avail lowest fare obligation

3.2 International Flights
- Flights under 6 hours: Economy class mandatory
- Flights 6 hours and above: Business class permitted with Director-level approval
- Business class requires written approval before ticket purchase
- First class: Never reimbursable (even if business class unavailable, must take economy)
- Stopovers/layovers: Accommodation covered only if layover exceeds 8 hours

4. HOTEL/ACCOMMODATION LIMITS

4.1 Domestic Accommodation
- Metro cities (Mumbai, Delhi, Bengaluru/Bangalore, Chennai, Hyderabad, Kolkata, Pune):
  Maximum Rs. 8,000 per night including taxes
- Non-metro cities and all other locations: Maximum Rs. 5,000 per night including taxes
- Extended stay (beyond 7 nights): Requires Finance Controller pre-approval
- Airbnb/homestays: Allowed only with prior approval; same limits apply

4.2 International Accommodation
- Standard rate: USD 200 per night including taxes
- High-cost cities (London, New York, Tokyo, Singapore, Zurich): USD 280 per night
- Must book through company's travel portal or approved partner hotels where available

5. MEAL ALLOWANCES

5.1 Domestic Meal Allowance (per day): Rs. 1,500
- Breakfast: Rs. 300
- Lunch: Rs. 500
- Dinner: Rs. 700
- Unused meal allowance: Not carried forward to next day
- If company provides meals (conferences, client sites): allowance proportionally reduced

5.2 International Meal Allowance (per day): USD 60
- Breakfast: USD 15
- Lunch: USD 20
- Dinner: USD 25
- High-cost cities: USD 80 per day
- Alcoholic beverages: NOT included in meal allowance; personal expense

6. LOCAL TRANSPORTATION DURING TRAVEL
- Taxi/cab (company-approved apps like Ola/Uber): Reimbursable with receipt
- Personal vehicle use: Rs. 12 per kilometer (fuel reimbursement basis)
- Must submit odometer readings or GPS-verified distance for personal vehicle claims
- Rental cars: Requires pre-approval; economy segment only
- Auto-rickshaw/local public transport: Reimbursable with self-declaration (no receipt needed)

7. TRAVEL ADVANCE AND SETTLEMENT
- Travel advance can be requested up to 7 days before travel departure
- Maximum advance: 80% of estimated travel cost
- Advance settlement deadline: Within 7 working days of return from travel
- Unutilized advance must be returned via company payment portal or deposited to cashier
- Interest charged at 12% per annum on advances outstanding beyond 15 days

8. INTERNATIONAL TRAVEL SPECIFICS
- Visa fees: Fully reimbursable with original receipt
- Travel insurance: Mandatory for international travel; arranged by company
- Foreign exchange: Employees may use Forex card or personal card (international transaction fees reimbursed)
- International SIM / data roaming allowance: Rs. 500 per day (maximum Rs. 7,500 per trip)
- Vaccinations required for destination country: Fully reimbursable
- Embassy attestation fees: Reimbursable if required for official travel

9. TRAVEL INSURANCE
- Company provides group travel insurance for all international travel
- Coverage includes: Medical emergency, trip cancellation, baggage loss, personal accident
- Coverage limit: USD 1,00,000 medical + USD 5,000 baggage
- Employees must inform HR before departure to activate insurance

10. NON-REIMBURSABLE TRAVEL EXPENSES
- Personal side trips during business travel
- Spouse/dependent travel (unless specifically approved under relocation policy)
- Excess baggage charges due to personal items
- In-flight entertainment, Wi-Fi (except for productivity needs on flights >4 hours)
- Minibar charges at hotels
- Lost baggage contents (covered under travel insurance separately)

11. CONTACT AND ESCALATIONS
- Travel desk: travel@company.com | Ext: 4422
- After-hours travel emergency: +91-9988776655
- Visa assistance: visa-support@company.com
"""

# ─────────────────────────────────────────────────────────────────────────
# DOCUMENT 3: Vendor Payment Procedures
# ─────────────────────────────────────────────────────────────────────────
doc3_text = """
VENDOR PAYMENT PROCEDURES
Policy Code: FIN-VPP-003 | Version: 4.1 | Effective Date: April 1, 2024
Issued by: Accounts Payable Team | Approved by: CFO

1. PURPOSE AND SCOPE
This document establishes the standard operating procedures for processing vendor
payments, ensuring accuracy, compliance, and timely settlements.

2. INVOICE REQUIREMENTS
All vendor invoices submitted for payment must include:
- Valid Purchase Order (PO) number issued by the Procurement team
- GST Invoice compliant with GST Act (GSTIN of both vendor and company)
- Delivery Challan or Goods Receipt Note (GRN) for physical goods
- Service Completion Certificate (SCC) for services — signed by receiving department
- Bank account details (if first invoice or bank details have changed)
- Invoice date, invoice number, payment terms clearly stated
- TDS applicability declaration from vendor

3. APPROVAL HIERARCHY FOR VENDOR PAYMENTS

Tier 1 — Up to Rs. 50,000:
- Approved by: Department Manager
- Processing time: 3 working days post approval

Tier 2 — Rs. 50,001 to Rs. 5,00,000 (5 lakhs):
- Approved by: Business Unit Director
- Processing time: 5 working days post approval

Tier 3 — Rs. 5,00,001 to Rs. 25,00,000 (25 lakhs):
- Approved by: Chief Financial Officer (CFO)
- Processing time: 7 working days post approval

Tier 4 — Above Rs. 25,00,000 (25 lakhs):
- Approved by: Board of Directors (requires Board resolution or designated committee)
- Processing time: As per Board meeting schedule

4. PAYMENT TERMS
- Domestic vendors: Net 30 days from invoice receipt date (standard)
- International vendors: Net 45 days from invoice receipt date (standard)
- Early payment discount: If vendor offers 2/10 Net 30, Finance may opt for early payment
- Advance payments: Only for verified vendors with specific contractual terms
- Advance payment limit: Maximum 25% of total PO value without CFO approval
- Retention amount: 10% withheld for project-based contracts until completion certificate

5. THREE-WAY MATCHING PROCESS
Before payment is processed, Accounts Payable performs mandatory 3-way matching:
Step 1 — PO Match: Invoice quantities and rates match Purchase Order
Step 2 — Invoice Match: Invoice details match received goods/services specification
Step 3 — GRN Match: Goods Receipt Note confirms physical receipt of goods/services
Any discrepancy in 3-way match triggers automatic payment hold and dispute process.

6. NEW VENDOR ONBOARDING REQUIREMENTS
Before any payment to a new vendor, the following documents must be collected:
- Certificate of Incorporation / Business Registration
- GST Registration Certificate (GSTIN)
- PAN Card of the company/proprietor
- Cancelled cheque or bank certificate for payment details
- MSME Registration (if applicable) — impacts TDS treatment
- Vendor Code must be created in ERP before any PO is raised
- Vendor due diligence check via approved vendor screening tool

7. TDS (TAX DEDUCTED AT SOURCE) REQUIREMENTS
TDS must be deducted as per Income Tax Act before releasing payment:
- Contractors and sub-contractors (Section 194C): 2% (companies), 1% (individuals/HUF)
- Professional services — CA, legal, technical (Section 194J): 10%
- Rent on equipment/machinery (Section 194I): 10%
- Commission payments (Section 194H): 5%
- TDS certificates (Form 16A) issued quarterly to vendors
- TDS returns filed quarterly: Q1 by July 31, Q2 by October 31, Q3 by January 31, Q4 by May 31

8. DISPUTE RESOLUTION PROCESS
- If vendor raises dispute on payment: Must be acknowledged within 2 working days
- Accounts Payable team initiates investigation immediately upon dispute receipt
- Internal resolution target: Within 10 working days
- Escalation to Finance Controller if unresolved within 10 days
- Formal dispute letter issued to vendor with resolution timeline
- Legal team involved only if dispute value exceeds Rs. 10 lakhs or is unresolved after 30 days

9. PAYMENT MODES
- NEFT/RTGS: Primary mode for domestic payments above Rs. 50,000
- IMPS: For urgent domestic payments below Rs. 2 lakh
- Wire transfer (SWIFT): For international vendor payments
- Cheque: Only for vendors without bank account (with Finance Head approval)
- Payment runs: Every Tuesday and Friday for standard payments

10. CONTACT INFORMATION
- Accounts Payable: accounts-payable@company.com | Ext: 4433
- Vendor queries: vendor-support@company.com
- New vendor onboarding: vendor-onboarding@company.com
"""

# ─────────────────────────────────────────────────────────────────────────
# DOCUMENT 4: Tax Compliance Guidelines
# ─────────────────────────────────────────────────────────────────────────
doc4_text = """
TAX COMPLIANCE GUIDELINES
Policy Code: FIN-TCG-004 | Version: 2.5 | Effective Date: April 1, 2024
Issued by: Taxation Department | Approved by: CFO & Company Secretary

1. PURPOSE
This document provides a comprehensive guide to the company's tax compliance
obligations under Indian tax laws, including direct and indirect taxes.

2. TDS (TAX DEDUCTED AT SOURCE) — RATES AND APPLICABILITY

2.1 TDS on Payments to Contractors (Section 194C)
- Rate for companies: 2% of payment amount
- Rate for individuals/HUF: 1% of payment amount
- Threshold for TDS deduction: Single payment above Rs. 30,000 OR
  aggregate payments in a financial year above Rs. 1,00,000
- Must deduct TDS before releasing any contractor payment above threshold

2.2 TDS on Professional Services (Section 194J)
- Rate: 10% for fees for professional services
- Applicable to: CA firms, lawyers, doctors, engineers, consultants, technical services
- Threshold: Single or aggregate payment above Rs. 30,000 in a financial year
- No TDS if vendor furnishes Form 15G/15H (only individuals below tax threshold)

2.3 TDS on Rent (Section 194I)
- Rent on land, building, furniture: 10%
- Rent on plant and machinery, equipment: 2%
- Threshold: Annual rent above Rs. 2,40,000

2.4 TDS on Employee Salaries (Section 192)
- TDS computed based on individual tax slab after all deductions
- Adjusted monthly based on projected annual income
- Employees must submit investment declaration by April 30 each year
- Revised declaration allowed once: by December 31

3. GST (GOODS AND SERVICES TAX) COMPLIANCE

3.1 Monthly GST Returns
- GSTR-1 (Outward supplies): Filed by 11th of following month
- GSTR-3B (Summary return + tax payment): Filed by 20th of following month
- For turnover above Rs. 5 crore: Monthly filing mandatory
- For turnover below Rs. 5 crore: Quarterly option available (QRMP scheme)

3.2 Annual GST Return
- GSTR-9 (Annual return): Filed by December 31 for previous financial year
- GSTR-9C (Reconciliation statement): If turnover above Rs. 5 crore, with CA certification

3.3 E-Invoicing Requirements
- Mandatory for businesses with turnover above Rs. 10 crore
- All B2B invoices must be generated through Invoice Registration Portal (IRP)
- IRN (Invoice Reference Number) must appear on every GST invoice
- E-invoice required before goods are dispatched; no retrospective generation

3.4 Input Tax Credit (ITC)
- ITC on business purchases claimable only if vendor has filed GSTR-1
- ITC claim must be matched with GSTR-2B before filing GSTR-3B
- ITC not available on: personal expenses, motor vehicles (unless resale/transport), food

4. EMPLOYEE TAX COMPLIANCE

4.1 Investment Declaration
- Deadline: April 30 of each financial year (beginning of year)
- Submit Form 12BB to HR/Payroll team
- Covers: HRA exemption, LTA exemption, Section 80C (PF, ELSS, NSC, insurance),
  Section 80D (health insurance), housing loan interest deduction
- Provisional declaration accepted initially; actual proofs required by January 31

4.2 HRA (House Rent Allowance) Exemption
- Requires: Rent receipts with landlord signature + PAN of landlord (if annual rent > Rs. 1 lakh)
- Notarized rent agreement recommended for rent above Rs. 8,000/month
- HRA exemption calculated as minimum of: actual HRA received, 50% of salary (metro)/40% (non-metro), actual rent minus 10% of salary

4.3 Form 16 — Annual Tax Certificate
- Issued by company to all employees by June 15 each year
- Part A: TDS details from Form 26AS
- Part B: Salary breakup and deductions summary
- Digital Form 16 available on HRMS portal; physical copy on request

5. ADVANCE TAX — QUARTERLY PAYMENT SCHEDULE
- 15% by June 15 (Q1)
- 45% by September 15 (cumulative) (Q2)
- 75% by December 15 (cumulative) (Q3)
- 100% by March 15 (cumulative) (Q4)
- Applicable if total tax liability exceeds Rs. 10,000 in the financial year
- Interest under Section 234B/234C if advance tax is under-paid

6. CORPORATE INCOME TAX
- Income tax return filed by October 31 (with audit) / July 31 (without audit)
- Transfer pricing documentation required for international transactions above Rs. 1 crore
- MAT (Minimum Alternate Tax) at 15% applies if regular tax is lower

7. CONTACT INFORMATION
- Tax team: tax-compliance@company.com | Ext: 4466
- TDS certificates: tds-certificates@company.com
- GST queries: gst-team@company.com
"""

# ─────────────────────────────────────────────────────────────────────────
# DOCUMENT 5: Procurement Approval Policy
# ─────────────────────────────────────────────────────────────────────────
doc5_text = """
PROCUREMENT APPROVAL POLICY
Policy Code: FIN-PAP-005 | Version: 3.0 | Effective Date: April 1, 2024
Issued by: Procurement Team | Approved by: CFO & CTO (IT Purchases)

1. PURPOSE AND SCOPE
This policy defines the approval authority matrix, procurement procedures, and
vendor selection criteria for all company purchases of goods and services.
Applies to: All departments, all purchase categories including IT, infrastructure, HR, operations.

2. PROCUREMENT AUTHORITY MATRIX

Tier 1 — Up to Rs. 25,000:
- Approver: Department Manager
- PO required: No (can use petty cash or credit card with receipt)
- Quotes required: 1 quote (verbal acceptable)
- Processing time: 1-2 working days

Tier 2 — Rs. 25,001 to Rs. 1,00,000 (1 lakh):
- Approver: Vice President (VP) of the department
- PO required: Yes
- Quotes required: Minimum 2 written quotes
- Processing time: 3-5 working days

Tier 3 — Rs. 1,00,001 to Rs. 10,00,000 (10 lakhs):
- Approvers: CFO + Department Head (dual approval required)
- PO required: Yes
- Quotes required: Minimum 3 written quotes from pre-qualified vendors
- Comparative analysis report (CAR) mandatory
- Processing time: 7-10 working days

Tier 4 — Above Rs. 10,00,000 (10 lakhs):
- Approvers: Board of Directors / Procurement Committee
- PO required: Yes (after Board approval)
- RFP (Request for Proposal) process mandatory
- Minimum 3 vendors in RFP process
- Processing time: 15-30 working days

SPECIAL RULE — IT Purchases (All Values):
- ALL IT purchases (hardware, software, licenses, SaaS subscriptions) require CTO approval
  regardless of purchase amount, in ADDITION to the standard tier approval
- IT security review mandatory for any software/SaaS tool above Rs. 50,000
- Cloud infrastructure spends: CTO + CFO dual approval above Rs. 5 lakh/month

3. MANDATORY QUOTE REQUIREMENTS
- Purchases above Rs. 1,00,000: Minimum 3 competitive quotes mandatory
- Quotes must be dated within 30 days of PO creation
- Quotes must be on vendor letterhead with valid GST number
- Sole-source justification required if only one vendor is available
- Sole-source must be approved by CFO and documented in procurement file

4. VENDOR SELECTION CRITERIA (Weighted Scoring Model)
When evaluating competing vendors, use the following scoring matrix:
- Price / Commercial Terms: 40% weightage
- Quality of product/service and technical specifications: 30% weightage
- Delivery timeline and fulfillment capability: 20% weightage
- Vendor reliability, references, financial stability: 10% weightage
Total score determines recommended vendor; lowest price alone is NOT sufficient criterion.

5. EMERGENCY PROCUREMENT PROCEDURE
Applicable when: Business operations are critically impacted and standard process timeline
cannot be followed.
- Step 1: CFO verbal approval obtained first (email/call/WhatsApp — all documented)
- Step 2: Written approval must be obtained within 24 hours of verbal approval
- Step 3: Retrospective PO raised within 2 working days of purchase
- Step 4: Post-purchase justification report submitted to Finance within 5 days
- Maximum emergency procurement value: Rs. 5,00,000 per instance
- Frequency limit: Maximum 3 emergency procurements per quarter per department
- Repeated emergency procurement indicates process failure; subject to audit review

6. PURCHASE ORDER (PO) MANAGEMENT
- PO validity: 90 days from date of issue
- PO amendments: Require original approver re-approval if value increases by more than 10%
- PO cancellation: Written notice to vendor with 7-day notice period
- Open POs older than 90 days automatically expired; must be renewed with fresh approvals
- Verbal commitments to vendors do NOT constitute a valid PO

7. CONFLICT OF INTEREST
- Employees must disclose if they have personal/financial relationship with any vendor
- Disclosure form submitted to HR and Finance before vendor engagement
- Recusal required from vendor evaluation and approval if conflict exists
- Violation of conflict of interest policy: Serious misconduct — disciplinary action

8. GOODS RECEIPT AND ACCEPTANCE
- GRN (Goods Receipt Note) must be signed by receiving department within 2 days of delivery
- Quality inspection report required for goods above Rs. 5 lakh
- Partial deliveries: GRN raised for quantities received; payment released proportionally
- Returns and rejections: Vendor must be notified within 5 working days

9. CONTACT INFORMATION
- Procurement team: procurement@company.com | Ext: 4477
- RFP coordination: rfp-team@company.com
- Vendor disputes: vendor-disputes@company.com
"""

# ─────────────────────────────────────────────────────────────────────────
# DOCUMENT 6: Employee Finance Handbook
# ─────────────────────────────────────────────────────────────────────────
doc6_text = """
EMPLOYEE FINANCE HANDBOOK
Policy Code: FIN-EFH-006 | Version: 5.0 | Effective Date: April 1, 2024
Issued by: HR & Finance | Approved by: CHRO & CFO

1. PURPOSE
This handbook is a comprehensive guide for employees covering salary structure,
payroll processes, financial benefits, loans, and how to seek finance support.

2. SALARY PAYMENT
- Salary credit date: Last working day of every month
- If last day is a bank holiday: Salary credited on the previous working day
- Mode: Direct bank transfer (NEFT/RTGS) to employee's registered salary account
- Payslip availability: HRMS portal → My Pay → Payslips (by 28th of each month)
- Salary revision cycle: Annual (April) — effective from April 1
- Mid-year corrections: Processed by Finance team within 30 days of HR approval

3. CTC (COST TO COMPANY) STRUCTURE
Standard CTC breakup for all employees:
- Basic Salary: 40% of CTC
- House Rent Allowance (HRA): 20% of CTC
- Special Allowance: 30% of CTC (taxable, flexible usage)
- Employer's Provident Fund (EPF) contribution: 12% of Basic Salary (employer share)
- Note: Employee PF contribution (12% of Basic) is deducted from gross salary
- Gratuity (from company): 4.81% of Basic Salary (payable after 5 years of service)
- Medical insurance: Premium paid by company; coverage Rs. 5 lakh/family

4. PAYROLL DEDUCTIONS
Monthly deductions from gross salary:
- Employee PF: 12% of Basic Salary
- Professional Tax: As per state government regulations (varies by state)
- Income Tax (TDS): Based on declared investments and tax slab
- Group insurance premium: If employee opts for enhanced coverage
- Loan/advance EMI: If employee has outstanding salary advance or personal loan

5. SALARY ADVANCE POLICY
- Eligibility: All confirmed employees (minimum 6 months' tenure)
- Maximum advance amount: Up to 2 months' net salary
- Maximum frequency: 2 times per calendar year
- Repayment: Equal EMIs over 6 months (auto-deducted from salary)
- Interest: No interest charged on salary advances
- Approval: HR Manager + Finance Manager dual approval required
- Application: HRMS portal → Finance → Salary Advance Request
- Outstanding advance: Cannot apply for second advance until first is fully repaid

6. PERSONAL LOAN SCHEME
- Eligibility: Employees with 3 or more years of continuous service
- Maximum loan amount: Rs. 2,00,000 (2 lakhs)
- Interest rate: 0% (interest-free loan from company)
- Repayment tenure: Up to 24 months (EMI from salary)
- Approval: HR Head + Finance Controller dual approval
- Purpose: Any personal financial need (medical, education, family emergency)
- Bond: Employee to sign loan agreement; recovery initiated if employee exits
- On resignation/termination: Outstanding loan balance deducted from Full & Final settlement

7. FULL AND FINAL SETTLEMENT (FnF)
- Trigger: Employee resignation, termination, or retirement
- Timeline: FnF settlement within 45 days of last working day
- Components: Unpaid salary + Earned leave encashment + Gratuity (if eligible)
  + Bonus (pro-rated) - outstanding loans/advances - any company dues
- Gratuity: Payable after 5 years of continuous service; formula: 15/26 x Basic x Years of Service
- Leave encashment: Up to 30 days of accumulated leave (beyond 30 days, lapses)

8. FINANCIAL BENEFITS AND ENTITLEMENTS
- Leave Travel Allowance (LTA): Once in 2 years; economy class air/train travel for self and family
- Medical reimbursement: Up to Rs. 15,000/year for out-of-pocket medical expenses
- Food coupons/sodexo: Rs. 2,200/month (tax-exempt up to applicable limits)
- Telephone reimbursement: Rs. 2,000/month for Grade 5 and above employees
- Vehicle maintenance allowance: Rs. 3,000/month for employees in field roles

9. KEY FINANCE CONTACTS FOR EMPLOYEES
- General reimbursements: finance-reimbursements@company.com | Ext: 4455
- Accounts payable: accounts-payable@company.com | Ext: 4433
- Payroll queries: payroll@company.com | Ext: 4444
- Tax and TDS queries: tax-compliance@company.com | Ext: 4466
- Travel advance: travel-advance@company.com | Ext: 4422

10. EMPLOYEE ESCALATION MATRIX FOR FINANCE QUERIES
If your finance query is not resolved, escalate as follows:
- Level 1: Finance Point of Contact (POC) — Resolution within 5 working days
- Level 2: Finance Manager — If unresolved at Level 1, escalate within 3 working days
- Level 3: Finance Controller — If unresolved at Level 2, escalate within 2 working days
- Level 4 (Final): CFO — Only for unresolved critical matters; resolution by next week
- HR Business Partner must be looped in for any Level 3+ escalation

11. IMPORTANT DEADLINES FOR EMPLOYEES
- Investment declaration: April 30 (beginning of financial year)
- Actual investment proofs submission: January 31
- Expense reimbursement submission: Within 30 days of expense
- Travel advance settlement: Within 7 working days of return
- Tax-saving investments (Section 80C): March 31 (end of financial year)
"""

# ── Create LangChain Document objects with metadata ────────────────────────
raw_documents = [
    Document(
        page_content=doc1_text.strip(),
        metadata={"source": "Employee Reimbursement Policy", "policy_code": "FIN-ERP-001"}
    ),
    Document(
        page_content=doc2_text.strip(),
        metadata={"source": "Corporate Travel Expense Policy", "policy_code": "FIN-CTR-002"}
    ),
    Document(
        page_content=doc3_text.strip(),
        metadata={"source": "Vendor Payment Procedures", "policy_code": "FIN-VPP-003"}
    ),
    Document(
        page_content=doc4_text.strip(),
        metadata={"source": "Tax Compliance Guidelines", "policy_code": "FIN-TCG-004"}
    ),
    Document(
        page_content=doc5_text.strip(),
        metadata={"source": "Procurement Approval Policy", "policy_code": "FIN-PAP-005"}
    ),
    Document(
        page_content=doc6_text.strip(),
        metadata={"source": "Employee Finance Handbook", "policy_code": "FIN-EFH-006"}
    ),
]

print(f"Finance policy documents loaded: {len(raw_documents)}")
for i, doc in enumerate(raw_documents, 1):
    word_count = len(doc.page_content.split())
    print(f"  Doc {i}: [{doc.metadata['policy_code']}] {doc.metadata['source']} — {word_count} words")
print("\nDocuments ready for chunking!")

In [ ]:
# Cell 6: Document Chunking
#
# RecursiveCharacterTextSplitter splits text hierarchically:
# First tries to split on paragraphs (\n\n), then lines (\n), then sentences,
# then words — preserving semantic coherence as much as possible.
#
# chunk_size=500  : Each chunk is ~500 characters (fits well in embedding context)
# chunk_overlap=50: 50-char overlap between chunks prevents information loss at boundaries

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    length_function=len,
    separators=["\n\n", "\n", ". ", " ", ""]  # Hierarchical split order
)

# Split all 6 documents into chunks; metadata (source, policy_code) is preserved per chunk
chunks = text_splitter.split_documents(raw_documents)

print(f"Chunking complete!")
print(f"  Total documents: {len(raw_documents)}")
print(f"  Total chunks created: {len(chunks)}")
print(f"  Average chunks per document: {len(chunks) / len(raw_documents):.1f}")
print()

# Show chunk distribution per document
from collections import Counter
chunk_sources = Counter(chunk.metadata['source'] for chunk in chunks)
print("Chunk distribution by document:")
for source, count in chunk_sources.items():
    print(f"  {count:3d} chunks — {source}")

# Display a sample chunk to verify quality
print(f"\n{'─'*60}")
print("SAMPLE CHUNK (chunk #5):")
print(f"{'─'*60}")
sample_chunk = chunks[4]
print(f"Source: {sample_chunk.metadata['source']}")
print(f"Length: {len(sample_chunk.page_content)} characters")
print(f"Content:\n{sample_chunk.page_content}")
print(f"{'─'*60}")

In [ ]:
# Cell 7: FAISS Vector Store Creation
#
# FAISS (Facebook AI Similarity Search) builds an efficient index for
# cosine/L2 similarity search over dense vector embeddings.
# Each chunk is embedded using text-embedding-3-small and stored in the index.

print("Building FAISS vector store (embedding all chunks)...")
print("This may take 30-60 seconds depending on API latency.")
print()

# Embed all chunks and build FAISS index
# FAISS.from_documents() calls the embeddings model for each chunk
vectorstore = FAISS.from_documents(chunks, embeddings)

# Create a retriever with similarity search, returning top-4 most relevant chunks
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4}   # Retrieve top-4 most relevant chunks per query
)

# Report index statistics
index_size = vectorstore.index.ntotal  # Total vectors in FAISS index
print(f"FAISS vector store built successfully!")
print(f"  Vectors indexed: {index_size}")
print(f"  Embedding model: text-embedding-3-small")
print(f"  Embedding dimensions: 1536")
print(f"  Retrieval: Top-4 chunks per query (similarity search)")
print(f"  Search type: Cosine similarity")
print()
print("Vector store ready for queries!")

In [ ]:
# Cell 8: RAG Chain Setup
#
# Builds the complete RetrievalQA chain using LangChain.
# The custom prompt template ensures the LLM answers ONLY from retrieved context
# and does not hallucinate finance policy information.

# ── Custom Finance Policy Prompt Template ─────────────────────────────────
FINANCE_PROMPT = PromptTemplate(
    input_variables=["context", "question"],
    template="""You are an AI-Powered Finance Policy Assistant for an enterprise organization.
Your job is to answer employee questions based ONLY on the official finance policy documents provided below.

IMPORTANT RULES:
- Answer ONLY based on the retrieved context below
- If the answer is not in the context, say: "This information is not covered in the current policy documents. Please contact finance-reimbursements@company.com"
- Be specific with amounts, limits, and procedures
- Do NOT generate information not present in the documents

Retrieved Policy Context:
{context}

Employee Question: {question}

Policy-Grounded Answer:"""
)

# ── Build RetrievalQA Chain ────────────────────────────────────────────────
# chain_type="stuff": Stuffs all retrieved chunks into a single prompt
# return_source_documents=True: Returns which chunks were used to answer
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    chain_type_kwargs={"prompt": FINANCE_PROMPT},
    return_source_documents=True
)

# ── Query Function ─────────────────────────────────────────────────────────
def query_rag(question: str):
    """
    Query the Finance Policy RAG system.

    Args:
        question (str): Natural language question from the employee.

    Returns:
        tuple: (answer: str, sources: list[Document])
    """
    # Invoke the chain
    result = qa_chain.invoke({"query": question})

    answer = result["result"]
    sources = result["source_documents"]

    # ── Pretty-print result ────────────────────────────────────────────────
    print("=" * 65)
    print(f"QUESTION: {question}")
    print("=" * 65)
    print(f"\nANSWER:")
    # Wrap long lines for readability
    for line in answer.split("\n"):
        if line.strip():
            print(textwrap.fill(line, width=65))
        else:
            print()

    print("\nSOURCE REFERENCES:")
    seen = set()
    for doc in sources:
        src = doc.metadata.get("source", "Unknown")
        if src not in seen:
            seen.add(src)
            snippet = doc.page_content[:120].replace("\n", " ")
            print(f"  [{src}]")
            print(f"   └─ {snippet}...")
    print()

    return answer, sources

print("RAG chain initialized successfully!")
print("  LLM: GPT-4o-mini (temperature=0)")
print("  Chain type: stuff (all chunks in single prompt)")
print("  Source documents: Returned with each query")
print("  Custom prompt: Finance-policy-grounded template active")
print()
print("query_rag() function is ready. Running a quick smoke test...")
print()

# Quick smoke test
_, _ = query_rag("What is the maximum reimbursement for domestic daily allowance?")

In [ ]:
# Cell 9: RAG Architecture Diagram
# Draws a professional flowchart of the complete RAG pipeline

fig, ax = plt.subplots(1, 1, figsize=(18, 11))
ax.set_xlim(0, 18)
ax.set_ylim(0, 11)
ax.axis('off')

# Color scheme
COLOR_DOC    = '#2196F3'   # Blue   — Documents
COLOR_CHUNK  = '#FF9800'   # Orange — Chunker
COLOR_EMBED  = '#FF5722'   # Deep Orange — Embeddings
COLOR_FAISS  = '#4CAF50'   # Green  — FAISS
COLOR_LLM    = '#F44336'   # Red    — LLM
COLOR_OUTPUT = '#9C27B0'   # Purple — Output
COLOR_QUERY  = '#607D8B'   # Blue-grey — Query
COLOR_BG     = '#FAFAFA'   # Light background
COLOR_DIVIDER= '#90A4AE'   # Divider line

# Helper: draw a rounded box with label
def draw_box(ax, x, y, w, h, label, sublabel, color, text_color='white', fontsize=9):
    box = FancyBboxPatch(
        (x - w/2, y - h/2), w, h,
        boxstyle="round,pad=0.12",
        facecolor=color, edgecolor='white', linewidth=2,
        zorder=3
    )
    ax.add_patch(box)
    ax.text(x, y + 0.13, label, ha='center', va='center',
            fontsize=fontsize, fontweight='bold', color=text_color, zorder=4)
    if sublabel:
        ax.text(x, y - 0.28, sublabel, ha='center', va='center',
                fontsize=7, color=text_color, alpha=0.9, zorder=4, style='italic')

# Helper: draw arrow
def draw_arrow(ax, x1, y1, x2, y2, color='#455A64', label=''):
    ax.annotate(
        '', xy=(x2, y2), xytext=(x1, y1),
        arrowprops=dict(
            arrowstyle='->', color=color, lw=2.0,
            connectionstyle='arc3,rad=0'
        ),
        zorder=5
    )
    if label:
        mx, my = (x1 + x2) / 2, (y1 + y2) / 2
        ax.text(mx, my + 0.18, label, ha='center', va='bottom',
                fontsize=7, color='#546E7A', zorder=6)

# ── Title ─────────────────────────────────────────────────────────────────
ax.text(9, 10.5, 'AI-Powered Finance Policy Assistant — RAG Architecture',
        ha='center', va='center', fontsize=15, fontweight='bold', color='#1A237E')

# ── Section labels ─────────────────────────────────────────────────────────
ax.text(3.8, 9.7, 'INGESTION PIPELINE  (One-time Setup)',
        ha='center', va='center', fontsize=10, color='#37474F',
        fontweight='bold',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='#E3F2FD', edgecolor='#90CAF9', alpha=0.9))

ax.text(13.2, 9.7, 'QUERY PIPELINE  (Real-time Inference)',
        ha='center', va='center', fontsize=10, color='#37474F',
        fontweight='bold',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='#FFF3E0', edgecolor='#FFCC80', alpha=0.9))

# ── Divider ───────────────────────────────────────────────────────────────
ax.plot([9, 9], [1.0, 9.3], '--', color=COLOR_DIVIDER, linewidth=1.8, alpha=0.6, zorder=2)
ax.text(9, 0.7, 'FAISS Vector Store  (Shared)', ha='center', va='center',
        fontsize=8, color='#37474F', style='italic')

# ─────────────────────────────────────────────────────────────────────────
# INGESTION PIPELINE (left side)
# ─────────────────────────────────────────────────────────────────────────

# 1. Finance Policy Documents
draw_box(ax, 1.6, 8.2, 2.6, 1.1,
         'Finance Policy Docs', '6 documents (synthetic)', COLOR_DOC)

# Doc list below
doc_labels = [
    'Employee Reimbursement Policy',
    'Corporate Travel Expense Policy',
    'Vendor Payment Procedures',
    'Tax Compliance Guidelines',
    'Procurement Approval Policy',
    'Employee Finance Handbook',
]
for i, label in enumerate(doc_labels):
    ax.text(1.6, 7.25 - i * 0.27, f'• {label}', ha='center', va='center',
            fontsize=6.5, color='#1565C0')

# 2. RecursiveCharacterTextSplitter
draw_box(ax, 1.6, 5.3, 2.7, 1.0,
         'Document Chunker', 'chunk_size=500, overlap=50', COLOR_CHUNK)
ax.text(1.6, 4.85, 'RecursiveCharacterTextSplitter', ha='center', va='center',
        fontsize=6.5, color='#BF360C', style='italic')

# 3. Embeddings
draw_box(ax, 1.6, 3.7, 2.6, 0.9,
         'text-embedding-3-small', 'OpenAI Embeddings API', COLOR_EMBED)
ax.text(1.6, 3.28, '1536-dim dense vectors', ha='center', va='center',
        fontsize=6.5, color='white', alpha=0.85)

# 4. FAISS Vector Store
draw_box(ax, 1.6, 2.2, 2.6, 0.9,
         'FAISS Vector Store', 'Similarity Index (cosine)', COLOR_FAISS)
ax.text(1.6, 1.78, 'In-memory  |  faiss-cpu', ha='center', va='center',
        fontsize=6.5, color='white', alpha=0.85)

# Ingestion arrows
draw_arrow(ax, 1.6, 7.65, 1.6, 5.82, label='Raw text')
draw_arrow(ax, 1.6, 4.80, 1.6, 4.17, label='Chunks')
draw_arrow(ax, 1.6, 3.25, 1.6, 2.67, label='Vectors')

# ─────────────────────────────────────────────────────────────────────────
# QUERY PIPELINE (right side)
# ─────────────────────────────────────────────────────────────────────────

# 1. User Query
draw_box(ax, 12.0, 8.2, 2.6, 1.0,
         'Employee Query', 'Natural language question', COLOR_QUERY)
ax.text(12.0, 7.75, '"What is the hotel limit in Bengaluru?"',
        ha='center', va='center', fontsize=6.5, color='white', alpha=0.85, style='italic')

# 2. Query Embeddings
draw_box(ax, 12.0, 6.8, 2.6, 0.9,
         'text-embedding-3-small', 'Query Embedding', COLOR_EMBED)
ax.text(12.0, 6.38, 'Same model → same vector space', ha='center', va='center',
        fontsize=6.5, color='white', alpha=0.85)

# 3. FAISS Similarity Search
draw_box(ax, 12.0, 5.4, 2.6, 0.9,
         'Similarity Search', 'FAISS  |  Top-k=4 chunks', COLOR_FAISS)
ax.text(12.0, 4.98, 'Nearest neighbor lookup', ha='center', va='center',
        fontsize=6.5, color='white', alpha=0.85)

# 4. Retrieved Context
draw_box(ax, 12.0, 4.0, 2.6, 0.9,
         'Retrieved Context', '4 policy chunks + metadata', COLOR_CHUNK)
ax.text(12.0, 3.58, 'Injected into LLM prompt', ha='center', va='center',
        fontsize=6.5, color='white', alpha=0.85)

# 5. GPT-4o-mini
draw_box(ax, 12.0, 2.6, 2.6, 0.9,
         'GPT-4o-mini', 'LLM Reasoning (temp=0)', COLOR_LLM)
ax.text(12.0, 2.18, 'Context-grounded generation', ha='center', va='center',
        fontsize=6.5, color='white', alpha=0.85)

# 6. Answer + Sources
draw_box(ax, 12.0, 1.3, 2.8, 0.9,
         'Grounded Answer + Sources', 'Policy-accurate response', COLOR_OUTPUT)
ax.text(12.0, 0.88, 'With source document references', ha='center', va='center',
        fontsize=6.5, color='white', alpha=0.85)

# Query pipeline arrows
draw_arrow(ax, 12.0, 7.70, 12.0, 7.27, label='Question text')
draw_arrow(ax, 12.0, 6.35, 12.0, 5.87, label='Query vector')
draw_arrow(ax, 12.0, 4.95, 12.0, 4.47, label='Top-4 chunks')
draw_arrow(ax, 12.0, 3.55, 12.0, 3.07, label='Prompt + context')
draw_arrow(ax, 12.0, 2.15, 12.0, 1.77, label='Generated answer')

# ── Bridge arrow: FAISS feeds both pipelines ──────────────────────────────
ax.annotate(
    '', xy=(10.7, 5.4), xytext=(2.9, 2.2),
    arrowprops=dict(
        arrowstyle='->', color='#4CAF50', lw=2.0,
        connectionstyle='arc3,rad=-0.3'
    ),
    zorder=5
)
ax.text(7.0, 3.2, 'Index lookup', ha='center', va='center',
        fontsize=7, color='#2E7D32', style='italic')

# ── Legend ────────────────────────────────────────────────────────────────
legend_items = [
    (COLOR_DOC,    'Policy Documents'),
    (COLOR_CHUNK,  'Chunker / Context'),
    (COLOR_EMBED,  'Embeddings (OpenAI)'),
    (COLOR_FAISS,  'FAISS Vector Store'),
    (COLOR_LLM,    'GPT-4o-mini (LLM)'),
    (COLOR_OUTPUT, 'Final Answer'),
    (COLOR_QUERY,  'User Query'),
]
for i, (color, label) in enumerate(legend_items):
    x_pos = 9.5 + (i % 4) * 2.2
    y_pos = 0.35 - (i // 4) * 0.35
    patch = mpatches.Patch(color=color, label=label)
    ax.add_patch(FancyBboxPatch((x_pos - 0.25, y_pos - 0.13), 0.5, 0.26,
                                boxstyle='round,pad=0.05', facecolor=color,
                                edgecolor='white', zorder=3))
    ax.text(x_pos + 0.4, y_pos, label, va='center', fontsize=7, color='#37474F')

plt.tight_layout()
plt.savefig('rag_architecture.png', dpi=150, bbox_inches='tight',
            facecolor='white', edgecolor='none')
plt.show()
print("Architecture diagram saved as 'rag_architecture.png'")

In [ ]:
# Cell 10: 10 Demo Queries
# Runs all 10 queries across all 6 policy documents to demonstrate the RAG system.

print("#" * 65)
print("   FINANCE POLICY ASSISTANT — 10 DEMO QUERIES")
print("   Covering all 6 policy documents")
print("#" * 65)
print()

demo_questions = [
    # Q1 — Employee Reimbursement Policy
    "What is the reimbursement limit for client entertainment per person?",
    # Q2 — Vendor Payment Procedures
    "What approvals are required for vendor payments above 5 lakhs?",
    # Q3 — Corporate Travel Expense Policy
    "Can I claim international travel expenses? What is the daily allowance?",
    # Q4 — Procurement Approval Policy
    "What documents are mandatory for procurement requests above 1 lakh?",
    # Q5 — Corporate Travel Expense Policy
    "What is the hotel accommodation limit for business travel in Bangalore?",
    # Q6 — Employee Reimbursement Policy
    "How many days does an employee have to submit expense reports after travel?",
    # Q7 — Procurement Approval Policy
    "What is the process for emergency procurement and what is the limit?",
    # Q8 — Employee Reimbursement Policy (non-reimbursable)
    "Are personal entertainment expenses reimbursable?",
    # Q9 — Tax Compliance Guidelines
    "What is the TDS rate applicable for professional service payments?",
    # Q10 — Procurement Approval Policy (IT)
    "What approvals are needed to purchase IT software for the team?",
]

all_answers = []
for i, question in enumerate(demo_questions, 1):
    print(f"\n{'▶'*3} Query {i:2d} of {len(demo_questions)} {'◀'*3}")
    answer, sources = query_rag(question)
    all_answers.append({"question": question, "answer": answer, "sources": sources})

print()
print("=" * 65)
print(f"All {len(demo_questions)} demo queries completed successfully!")
print("=" * 65)

In [ ]:
# Cell 11: Interactive Q&A Interface
# Provides a live command-line interface for employees to ask custom questions.
# Type 'exit', 'quit', or 'q' to end the session.

print("=" * 65)
print("   Finance Policy Assistant — Interactive Mode")
print("   Ask any question about company finance policies.")
print("   Type 'exit' to quit")
print("=" * 65)
print()

while True:
    try:
        question = input("Your Question: ").strip()
    except (EOFError, KeyboardInterrupt):
        # Handle non-interactive environments gracefully
        print("\nInteractive mode ended.")
        break

    if question.lower() in ["exit", "quit", "q", ""]:
        print("Goodbye! For further assistance: finance-reimbursements@company.com")
        break

    if question:
        query_rag(question)

# Technical Documentation

---

## System Design

### RAG Pipeline Overview

Retrieval-Augmented Generation (RAG) combines two phases:

1. **Ingestion Phase** (one-time): Policy documents are chunked, embedded into dense vectors, and indexed in FAISS. This creates a searchable knowledge base from unstructured text.

2. **Query Phase** (real-time): When a user submits a question, it is embedded using the same model. The top-k most semantically similar chunks are retrieved from FAISS and injected into the LLM prompt as context. The LLM then generates a grounded answer based solely on the retrieved evidence.

### Why RAG Over Fine-Tuning?

| Dimension | RAG | Fine-Tuning |
|-----------|-----|-------------|
| Cost | Low (API calls only) | High (GPU training) |
| Update speed | Instant (add documents) | Slow (retrain) |
| Traceability | Source documents cited | Black-box |
| Hallucination risk | Low (context-grounded) | Higher |
| Accuracy on specific policies | High | Medium |
| Infrastructure | Minimal | GPU cluster needed |

RAG is the correct choice for enterprise policy assistants where **accuracy**, **traceability**, and **frequent document updates** are priorities.

---

## Chunking Strategy

**Splitter:** `RecursiveCharacterTextSplitter`

**Parameters:**
- `chunk_size = 500` characters: Balances specificity (small enough to be topic-focused) with context (large enough to contain a complete policy rule)
- `chunk_overlap = 50` characters: Prevents information loss at chunk boundaries — a rule that spans two chunks is captured by overlap

**Why Recursive?** The splitter first tries to split on paragraph breaks (`\n\n`), then line breaks, then sentences, then words. This preserves semantic boundaries (e.g., it avoids splitting mid-sentence), unlike a simple character splitter.

**Metadata preservation:** Each chunk inherits the `source` and `policy_code` fields from its parent document, enabling accurate source attribution in answers.

---

## Retrieval Approach

**Embedding model:** `text-embedding-3-small` (OpenAI)
- 1536-dimensional dense vectors
- Chosen for: high retrieval quality, cost efficiency, fast inference
- Both documents and queries use the same model → same vector space → accurate similarity

**Vector database:** FAISS (`faiss-cpu`)
- Flat L2 index for exact nearest-neighbor search (suitable for corpus of ~100-500 chunks)
- For larger corpora (>100K chunks), switch to `IndexIVFFlat` or `IndexHNSW`

**Search type:** Cosine similarity (semantic matching)

**Top-k = 4:** Retrieves 4 chunks per query. This provides enough context (~2000 chars) for GPT-4o-mini to answer accurately without exceeding prompt budget.

---

## Prompt Engineering

The custom `FINANCE_PROMPT` enforces three key behaviors:

1. **Grounding constraint:** "Answer ONLY based on the retrieved context" — prevents hallucination
2. **Graceful fallback:** If information is absent, direct to human contact rather than fabricating an answer
3. **Specificity instruction:** "Be specific with amounts, limits, and procedures" — ensures numeric precision for policy figures

---

## Limitations

| Limitation | Description |
|------------|-------------|
| Static knowledge | Documents are loaded once; no real-time policy updates |
| English only | System processes English queries; multilingual support not implemented |
| No memory | Each query is independent; no conversation history |
| Synthetic data | Documents are synthetic; replace with actual policy PDFs for production |
| No confidence scoring | System does not indicate certainty level of retrieved chunks |
| No PDF/DOCX ingestion | Current implementation uses raw text; production requires document loaders |

---

## Future Improvements

1. **Multi-language support:** Integrate translation layer or multilingual embedding model (`multilingual-e5-large`) for Hindi, Tamil, and other regional languages

2. **Document auto-update pipeline:** Connect to SharePoint/Confluence API to automatically re-index when policies are updated

3. **Conversation memory:** Add `ConversationBufferMemory` or `ConversationSummaryMemory` to support multi-turn dialogue ("What about the international version of that?") 

4. **Confidence scoring:** Implement retrieval score thresholds — if best chunk similarity < 0.7, return "Policy not found" instead of attempting to answer

5. **Hybrid search:** Combine BM25 (keyword) + FAISS (semantic) using `EnsembleRetriever` for improved recall on specific terms like policy codes and amounts

6. **PDF/DOCX ingestion:** Use `PyPDFLoader` or `Docx2txtLoader` to ingest actual company policy documents directly from file storage

7. **Feedback loop:** Collect thumbs-up/thumbs-down signals per query to fine-tune retrieval and prompt quality over time

8. **Persistent FAISS index:** Save/load the FAISS index to disk with `vectorstore.save_local()` / `FAISS.load_local()` to avoid re-indexing on every session start

---

## System Specifications

| Component | Specification |
|-----------|---------------|
| LLM | GPT-4o-mini (OpenAI) — temperature=0 |
| Embedding Model | text-embedding-3-small — 1536 dims |
| Orchestration | LangChain (RetrievalQA, LCEL) |
| Vector Store | FAISS (faiss-cpu) — flat L2 index |
| Chunk Size | 500 characters |
| Chunk Overlap | 50 characters |
| Top-k Retrieval | 4 chunks per query |
| Policy Documents | 6 synthetic finance policies |
| Runtime | Google Colab (Python 3.10+) |

---

*Notebook version 1.0 — AI-Powered Finance Policy Assistant using RAG*